# 03 · Pool projection & scenarios: the handoff to the waterfall

**Purpose.** Project the whole STACR reference pool month by month to the Feb 2031 call (53 months) under several scenarios, and produce the table that Smarajit's `src/waterfall.py` consumes. His waterfall only needs this table, not how CPR or defaults were modeled.

**Process (`src/collateral_projection.py`).**
1. **Scheduled principal**: level-pay amortization. Payment = B·r / (1 − (1+r)⁻ⁿ), and principal = payment − B·r. Checked by hand: $300k, 6.75%, 342 months gives month-1 principal of $290.46.
2. **`project_collateral`, one path.** Each month, in this order so no dollar leaves twice:
   1. *Defaults*: MDR × performing balance enter a 12-month **liquidation pipeline**. They stay in the reference pool, counted in `distressed_balance`, which feeds the STACR Delinquency Test. 25% of new defaults are **modified** instead and stay performing at a lower rate.
   2. *Scheduled principal* on the remaining performing balance.
   3. *Prepayments*: SMM × (balance after scheduled principal).
   4. *Credit events*: loans whose 12-month lag ends leave the pool, and **loss** = balance × severity at that month's HPI. Loans already 60+ days delinquent or in bankruptcy/foreclosure on the tape start in the pipeline and liquidate in month 6.

   Balance identity: ending = beginning − scheduled − prepayments − credit events (checked every month).
3. **`run_scenarios`**: good / base / moderate / severe paths for HPI and mortgage rates (**placeholders** until Coco delivers).
4. **Figure 3** and a summary of cumulative losses against the tranche attachment points.
5. **Save** each scenario's table to `outputs/pool_cf_<scenario>.parquet` for the waterfall.

Output columns: `month, beginning_balance, scheduled_principal, prepayments, defaults (credit events), losses, recoveries, ending_balance, modification_losses, distressed_balance`, all in $.

In [ ]:
import sys, warnings
from pathlib import Path

# Find the repo root (the folder that contains src/) so imports work from anywhere
ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)  # pandas 2.2 + Python 3.14 noise

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.float_format", "{:,.4f}".format)
print("repo root:", ROOT)

In [ ]:
def try_run(fn, *args, **kwargs):
    """Call a model function; if it's still a stub, say so instead of crashing."""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError:
        print(f"-> {fn.__module__}.{fn.__name__} is not implemented yet")
        return None

In [ ]:
from src import collateral_projection as cp
pool = pd.read_parquet(ROOT / "data" / "processed" / "stacr_dna1.parquet")
print(f"{len(pool):,} loans, ${pool['Current Balance'].sum()/1e9:.2f}bn")
print('output columns:', cp.COLUMNS)

## Step 1 · Scheduled principal

> Check by hand: $300k, 6.75%, 342 months left. What's month-1 principal?

In [ ]:
sp = try_run(cp.scheduled_principal, np.array([300_000.0]), np.array([6.75]), np.array([342]))
print(sp)

**Result · Scheduled principal.** **$290.46**, matching the hand check (payment $1,977.96 − interest $1,687.50).

## Step 2 · One path → monthly table

> Within a month, which balance does SMM apply to, and which does MDR? Does your table conserve balance?

In [ ]:
N_MONTHS = 53
flat_rate = np.full(N_MONTHS, 6.25)
flat_hpi = np.ones(N_MONTHS + 1)

cf = try_run(cp.project_collateral, pool, flat_hpi, flat_rate, N_MONTHS)
if cf is not None:
    out = cf["scheduled_principal"] + cf["prepayments"] + cf["defaults"]
    print("max balance error ($):", (cf["beginning_balance"] - out - cf["ending_balance"]).abs().max())
    display(cf.head(12))

**Result · One path (flat HPI, 6.25% rate).** The balance identity holds every month (max error < $0.00001), starting at **$19.44bn**.
- **Prepayments dominate**: about **$283mm** in month 1 vs **$19mm** of scheduled principal, so about 94% of principal comes from prepayments.
- **Credit events**: zero in months 1–5, then **$48mm in month 6**. Those are the loans already 60+ days late or coded B/F on the tape, liquidating halfway through the lag, with a **$3.2mm loss** (about 7% severity). New defaults only start liquidating in month 13.
- **`distressed_balance`** starts at **$51mm**, grows to about $64mm as new defaults enter the pipeline, drops to $19mm after the month-6 liquidations, then rebuilds. This column feeds the waterfall's Delinquency Test.
- **`modification_losses`** grow slowly from $0 to about $12k a month as modified loans accumulate.

## Step 3 · Scenarios

Placeholder paths; swap in Coco's from `src/scenarios.py`.

In [ ]:
scenarios = {
    "good":     {"hpi": np.linspace(1.00, 1.25, N_MONTHS + 1), "rate": np.full(N_MONTHS, 5.00)},
    "base":     {"hpi": np.linspace(1.00, 1.15, N_MONTHS + 1), "rate": np.full(N_MONTHS, 6.25)},
    "moderate": {"hpi": np.linspace(1.00, 0.90, N_MONTHS + 1), "rate": np.full(N_MONTHS, 5.75)},
    "severe":   {"hpi": np.linspace(1.00, 0.75, N_MONTHS + 1), "rate": np.full(N_MONTHS, 5.25)},
}
results = try_run(cp.run_scenarios, pool, scenarios, N_MONTHS)

**Result · Scenarios.** `run_scenarios` produced four 53-month tables (good, base, moderate, severe), one per placeholder path. Their results are below.

### Figure 3 · Projected reference-pool balance

In [ ]:
if results:
    for name, df in results.items():
        plt.plot(df["month"], df["ending_balance"] / 1e9, label=name)
    plt.xlabel("month"); plt.ylabel("pool balance ($bn)"); plt.legend()

**Result · Figure 3 (pool balance).** All four paths pay down fast, but at different speeds:

| | month 12 | month 24 | month 36 | month 53 | half paid down by |
|---|---|---|---|---|---|
| good | $12.0bn | $7.9bn | $5.9bn | **$4.41bn** | month 18 |
| base | $15.5bn | $12.3bn | $10.0bn | **$7.85bn** | month 39 |
| moderate | $13.5bn | $9.5bn | $7.2bn | **$5.26bn** | month 23 |
| severe | $12.2bn | $8.0bn | $5.9bn | **$4.34bn** | month 19 |

The **good** path (rates 5.00%) pays down fastest: CPR is about 37% in year 1, so half the pool is gone by month 18. The stressed paths also pay down **faster** than base, but only because the placeholder stress also **cuts rates** (5.75% and 5.25%), so prepayments speed up. In a real downturn, underwater borrowers cannot refinance, so this is too optimistic for the severe case.

In [ ]:
if results:
    display(pd.DataFrame({name: {
        "cum. losses ($mm)": df["losses"].sum() / 1e6,
        "cum. loss / cut-off (%)": df["losses"].sum() / cp.CUTOFF_BALANCE * 100,
        "end balance ($bn)": df["ending_balance"].iloc[-1] / 1e9,
    } for name, df in results.items()}))

**Result · Losses vs the tranche stack.** Cumulative losses (good / base / moderate / severe) are **$2.6mm / $3.7mm / $10.6mm / $21.4mm**, or **0.011% / 0.016% / 0.047% / 0.094%** of the $22.78bn cut-off balance. The first-loss piece **B-3H covers 0–0.25%**, so even the severe case uses only about 40% of B-3H. B-2H, B-1H and the offered **M-2B (attaching at 1.90%), M-1 and A-1 take no write-downs**.

> Compare cumulative loss / cut-off with the tranche bands in `docs/cashflows.md` (B-3H 0–0.25%, B-2H 0.25–1.45%, …, M-2B attaches at 1.90%). Which scenario reaches the offered notes?

## Step 4 · Save the handoff for Smarajit

In [ ]:
if results:
    out_dir = ROOT / "outputs"; out_dir.mkdir(exist_ok=True)
    for name, df in results.items():
        df.to_parquet(out_dir / f"pool_cf_{name}.parquet", index=False)
    print("saved to", out_dir)

**Result · Saved.** Four files are in `outputs/` (gitignored): `pool_cf_good.parquet`, `pool_cf_base.parquet`, `pool_cf_moderate.parquet` and `pool_cf_severe.parquet`, each with 53 rows and the 10 agreed columns. These are what Smarajit's waterfall reads.

## Results and takeaways (placeholder parameters and scenarios)

- **The table is consistent**: the balance identity holds every month (max error < $0.00001), and month 1 starts at today's **$19.44bn**.
- **Prepayment dominates the cash flow**: about $280–315mm a month of prepayments vs about $18mm of scheduled principal in the base case. CPR averages about **37% in year 1** (good), **19%** (base, 16–21%), 26–32% (moderate) and 33–38% (severe, because the placeholder stress also cuts rates).
- **Pool runoff to the Feb 2031 call**: the pool falls to **$4.41bn (good), $7.85bn (base), $5.26bn (moderate), $4.34bn (severe)**. Faster runoff shortens the notes' life (WAL), and Smarajit's waterfall turns this into A-1/M-1/M-2 principal.
- **Credit events**: about $48mm of loans that were already seriously delinquent liquidate in month 6 (a $3.2mm loss on the flat path in Step 2, $2.6–4.3mm across the scenarios). New defaults start liquidating after the 12-month lag. Total credit events are **$88mm / $101mm / $105mm / $121mm** (good / base / moderate / severe).
- **Losses**: cumulative losses are **$2.6mm / $3.7mm / $10.6mm / $21.4mm**, or **0.011% / 0.016% / 0.047% / 0.094%** (good / base / moderate / severe) of the $22.78bn cut-off balance. All of that stays inside **B-3H (0–0.25%)**, the first-loss piece Freddie Mac retains. **A-1, M-1 and M-2 take no write-downs** in these scenarios. M-2B would need cumulative losses above 1.90%.
- **Distressed balance** (for the Delinquency Test) peaks at about $64mm, far below the trigger threshold, so the triggers keep passing and principal goes pro rata between senior and subordinate.

**What this means and what's next.** With this low-LTV pool, the offered notes are mainly exposed to **prepayment and timing risk**, not credit losses. Losses only reach them under much harsher paths. Next: (1) calibrate PARAMS on the Freddie Single-Family Loan-Level Dataset; (2) block refinancing for underwater loans so severe scenarios don't prepay away the risk; (3) replace the placeholder paths with Coco's scenarios, including a severe path with *higher* unemployment and HPI −30% or worse.